# Step 4: Add Metadata and Analyze Spatial Patterns


This notebook turns metric results into spatial summaries. You will use PGA and FAS as two separate examples, so you can see how the same spatial tests can highlight different model-performance patterns for different metrics.


## Imports

These functions prepare metric fields and calculate spatial summaries.


In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
repo_root = runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from spatial_vtk.config import (
    notebook_timer,
    notebook_figure_settings,
    prepare_notebook_geospatial_environment,
    register_svtk_cell_timer,
)
prepare_notebook_geospatial_environment(loky_max_cpu_count=1)

with notebook_timer():

    from IPython.display import Markdown, display

    from spatial_vtk.config.labels import metric_display_name
    from spatial_vtk.io import load_configured_input_tables, output_group
    from spatial_vtk.spatial import (
        run_spatial_statistics_workflow_from_config,
        spatial_metric_product_summary_frame,
        spatial_workflow_failure_frame,
    )
    from spatial_vtk.spatial.map import plot_pca_summary, plot_residual_grid, plot_station_bias_map
    from spatial_vtk.spatial.plot import plot_distance_correlation_by_metric, plot_geology_contrast
    register_svtk_cell_timer()


## Configuration

Load the config and read the spatial-statistics settings from the tutorial run scenario.


In [ ]:
from spatial_vtk.config import notebook_figure_settings, notebook_run_context

config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it the active config for later package calls.
context = notebook_run_context(config_path, run_scenario="tutorial")
cfg = context.cfg
step_outputs = output_group("step_04_spatial", cfg=cfg)

# Step 4 uses the configured larger QC-passed metric snapshot so per-metric spatial tests have enough stations.
spatial_figure_settings = notebook_figure_settings("spatial")
spatial_sidecars = spatial_figure_settings.sidecars
spatial_metrics = ("PGA", "FAS")
add_basemap = spatial_figure_settings.add_basemap


## Prepare Metric-Specific Spatial Fields

Spatial statistics use one value per event-station observation, plus station and event coordinates. This notebook delegates the table-building work to the package workflow, then uses the returned tables for compact displays and figures.

In [ ]:
# Run the config-backed spatial workflow and then load the standard tables it writes.
# The metrics and station metadata arguments are config keys, so the notebook does not resolve paths itself.
configured_inputs = load_configured_input_tables({"site_metadata": "paths.site_metadata"}, cfg=cfg)
site_metadata = configured_inputs["site_metadata"]
spatial_result = run_spatial_statistics_workflow_from_config(
    config_path=str(config_path),
    run_scenario="tutorial",
    metrics="paths.metric_figure_snapshot",
    metric=spatial_metrics,
    station_metadata="paths.site_metadata",
    verbose=True,
)

failure_table = spatial_workflow_failure_frame(spatial_result)
if not failure_table.empty:
    display(Markdown("### Non-fatal workflow diagnostics"))
    display(failure_table)

# Keep small per-metric table handles for the plotting cells below.
spatial_tables = step_outputs.load_tables(
    {
        "metric_field": "metric_field_path",
        "event_centered_residuals": "event_centered_path",
        "station_bias": "station_bias_path",
        "morans_i": "morans_i_path",
        "distance_bins": "distance_corr_path",
        "clusters": "clusters_path",
        "cluster_scores": "cluster_scores_path",
        "cluster_summary": "cluster_summary_path",
        "pca_station_scores": "pca_scores_path",
        "pca_feature_loadings": "pca_loadings_path",
        "pca_explained_variance": "pca_explained_path",
        "geology_contrasts": "geology_path",
    },
    cfg=cfg,
)
metric_field = spatial_tables["metric_field"]
event_centered_residuals = spatial_tables["event_centered_residuals"]
station_bias = spatial_tables["station_bias"]
spatial_metrics_run = tuple(spatial_result["metrics"])
spatial_products = {}
for metric_name in spatial_metrics_run:
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    field = metric_field.loc[metric_field["metric"].astype(str).eq(metric_name)].copy()
    centered = event_centered_residuals.loc[event_centered_residuals["metric"].astype(str).eq(metric_name)].copy()
    bias = station_bias.loc[station_bias["metric"].astype(str).eq(metric_name)].copy()
    spatial_products[metric_name] = {"field": field, "centered": centered, "station_bias": bias}
    display(
        spatial_metric_product_summary_frame(
            metric_field=field,
            event_centered=centered,
            station_bias=bias,
        )
    )
    display(bias.head())


## Station Bias Maps

These maps show the mean event-centered residual at each station. Positive values mean the observed amplitudes are larger than the synthetic amplitudes on average for that metric.


In [ ]:
for metric_name, products in spatial_products.items():
    display(Markdown(f"### {metric_display_name(metric_name)}"))

    # Map the mean event-centered residual at each station for this metric.
    station_bias_fig = plot_station_bias_map(
        products["station_bias"],
        title=f"{metric_display_name(metric_name)} Station Bias",
        value_col="mean_centered",
        value_label="Mean event-centered log2(obs/syn)",
        add_basemap=add_basemap,
        showfig=True,
        savefig=True,
        outpath=step_outputs.figure_path("station_bias_figure_path", stem_parts=("step_04", metric_name, "station_bias")),
        **spatial_sidecars.kwargs(),
    )


## Residual Grid Maps

A residual grid gives you a quick spatial overview of where residuals are broadly positive or negative. These examples use the same event-centered residual field as the station-bias maps.


In [ ]:
for metric_name, products in spatial_products.items():
    display(Markdown(f"### {metric_display_name(metric_name)}"))

    # Grid event-centered residuals onto lon/lat cells for a broader spatial view.
    residual_grid_fig = plot_residual_grid(
        products["centered"],
        lon_col="lon",
        lat_col="lat",
        value_col="field_centered",
        cell_size_deg=0.05,
        title=f"{metric_display_name(metric_name)} Residual Grid",
        add_basemap=add_basemap,
        showfig=True,
        savefig=True,
        outpath=step_outputs.figure_path("residual_grid_figure_path", stem_parts=("step_04", metric_name, "residual_grid")),
        **spatial_sidecars.kwargs(),
    )


## Spatial Correlation Tests

Moran's I and distance-bin correlations help you see whether residuals cluster in space. The workflow has already written these tables; this cell displays them and plots the distance-binned correlations together.

In [ ]:
# Use the Moran's I and distance-bin correlation tables written by the spatial workflow.
morans_i = spatial_tables["morans_i"]
distance_bins = spatial_tables["distance_bins"]

for metric_name in spatial_metrics_run:
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    display(morans_i.loc[morans_i["metric"].astype(str).eq(metric_name)])
    display(distance_bins.loc[distance_bins["metric"].astype(str).eq(metric_name)].head())

# Plot correlation as a function of station separation distance for the selected metrics.
correlation_distance_fig = plot_distance_correlation_by_metric(
    distance_bins,
    significance_df=morans_i,
    title="Spatial Correlation by Distance",
    showfig=True,
    savefig=True,
    outpath=step_outputs.figure_path("spatial_correlation_distance_figure_path", stem_parts=("step_04", "spatial_correlation_distance")),
    **spatial_sidecars.kwargs(),
)


## Clustering and PCA Spatial Modes

These summaries group stations with similar residual fingerprints and identify dominant station-level patterns. The workflow writes the clustering and PCA tables; this cell renders PCA summary figures from those outputs.

In [ ]:
# Use the clustering and PCA tables written by the spatial workflow.
clusters = spatial_tables["clusters"]
cluster_scores = spatial_tables["cluster_scores"]
cluster_summary = spatial_tables["cluster_summary"]
pca_station_scores = spatial_tables["pca_station_scores"]
pca_feature_loadings = spatial_tables["pca_feature_loadings"]
pca_explained_variance = spatial_tables["pca_explained_variance"]

for metric_name in spatial_metrics_run:
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    station_scores = pca_station_scores.loc[pca_station_scores["metric"].astype(str).eq(metric_name)].copy()
    explained = pca_explained_variance.loc[pca_explained_variance["metric"].astype(str).eq(metric_name)].copy()
    loadings = pca_feature_loadings.loc[pca_feature_loadings["metric"].astype(str).eq(metric_name)].copy()

    # Plot the PC1 map, explained variance, and feature loading summary for this metric.
    pca_summary_fig = plot_pca_summary(
        station_scores,
        explained,
        loadings,
        mode="PC1",
        title=f"{metric_display_name(metric_name)} PCA Spatial Mode Summary",
        add_basemap=add_basemap,
        showfig=True,
        savefig=True,
        outpath=step_outputs.figure_path("pca_summary_figure_path", stem_parts=("step_04", metric_name, "pca_summary")),
        **spatial_sidecars.kwargs(),
    )
    display(explained)


## Geology Contrasts

Compare PGA and FAS residuals across the configured geology classes. GeoJSON regions and corridors are handled in the next notebook.

In [ ]:
# Use the geology contrast table written by the spatial workflow.
geology_contrasts = spatial_tables["geology_contrasts"]

for metric_name, products in spatial_products.items():
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    geology_contrast = geology_contrasts.loc[geology_contrasts["metric"].astype(str).eq(metric_name)].copy()

    # Plot the residual distributions and annotate the configured geology-class contrast for this metric.
    geology_contrast_fig = plot_geology_contrast(
        products["centered"],
        station_metadata=site_metadata,
        contrast_df=geology_contrast,
        title=f"{metric_display_name(metric_name)} Residuals by Geology Class",
        showfig=True,
        savefig=True,
        outpath=step_outputs.figure_path("geology_contrast_figure_path", stem_parts=("step_04", metric_name, "geology_contrast")),
        **spatial_sidecars.kwargs(),
    )
    display(geology_contrast)


## Spatial Figure Provenance

Review the figure sidecar metadata written for this step without loading the full row CSV sidecars.

In [ ]:
spatial_sidecars.status_frame()
